# 📚 Project 2 — DocuMind: RAG Agent with Citation Grounding

**Core Concept:** Retrieve relevant document chunks, generate grounded answers, return citations

### Architecture
User Question
      │
      ▼
Embed Question → Vector
      │
      ▼
Search ChromaDB
      │
      ▼
Retrieve Top 3 Chunks + Sources
      │
      ▼
LLM Generates Answer from Context
      │
      ▼
Return Answer + Citations + Confidence

Install

In [1]:
!pip install -q langchain langchain-groq langchain-community chromadb sentence-transformers loguru

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.

API Key

In [2]:
import os
os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"

Imports

In [3]:
import os
from loguru import logger
import sys
from typing import Optional
import chromadb
from chromadb.utils import embedding_functions
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

logger.remove()
logger.add(sys.stdout, format="{time:HH:mm:ss} | {level} | {message}", level="DEBUG")
print("Imports successful")

Imports successful


Sample Documents

In [4]:
documents = [
    {
        "id": "doc1",
        "content": """Artificial Intelligence (AI) is the simulation of human intelligence
        in machines programmed to think and learn. Machine learning is a subset of AI
        that enables systems to learn from data without explicit programming.
        Deep learning uses neural networks with many layers to process complex patterns.""",
        "source": "AI_Fundamentals.pdf",
        "page": 1
    },
    {
        "id": "doc2",
        "content": """Large Language Models (LLMs) are AI systems trained on massive text datasets.
        They can generate human-like text, answer questions, and perform reasoning tasks.
        Examples include GPT-4, Claude, and LLaMA. These models use transformer architecture
        with attention mechanisms to understand context.""",
        "source": "LLM_Guide.pdf",
        "page": 3
    },
    {
        "id": "doc3",
        "content": """Retrieval Augmented Generation (RAG) combines information retrieval
        with text generation. It first searches a knowledge base for relevant documents,
        then uses those documents as context for the LLM to generate accurate answers.
        RAG reduces hallucinations and improves factual accuracy significantly.""",
        "source": "RAG_Techniques.pdf",
        "page": 7
    },
    {
        "id": "doc4",
        "content": """Vector databases store data as high-dimensional vectors called embeddings.
        They enable semantic search — finding similar content based on meaning rather than
        exact keyword matching. Popular vector databases include ChromaDB, Pinecone, and Weaviate.
        They are essential infrastructure for RAG systems.""",
        "source": "Vector_DB_Guide.pdf",
        "page": 2
    },
    {
        "id": "doc5",
        "content": """Agentic AI refers to AI systems that can autonomously plan, reason,
        and take actions to complete complex tasks. Unlike simple chatbots, agents can
        use tools, maintain memory, and make multi-step decisions. LangChain and LangGraph
        are popular frameworks for building agentic AI systems.""",
        "source": "Agentic_AI.pdf",
        "page": 5
    },
    {
        "id": "doc6",
        "content": """Prompt engineering is the practice of designing inputs to AI models
        to get desired outputs. Techniques include chain-of-thought prompting, few-shot
        learning, and role prompting. Good prompt engineering can dramatically improve
        LLM performance without changing model weights.""",
        "source": "Prompt_Engineering.pdf",
        "page": 4
    }
]

logger.info(f"Loaded {len(documents)} documents")

04:06:00 | INFO | Loaded 6 documents


Initialize ChromaDB + Embed Documents

In [5]:
# Initialize ChromaDB with sentence transformers embedding
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

client = chromadb.Client()

# Delete collection if exists to avoid duplicates on rerun
try:
    client.delete_collection("documind")
except:
    pass

collection = client.create_collection(
    name="documind",
    embedding_function=embedding_fn
)

# Add documents to ChromaDB
collection.add(
    ids=[doc["id"] for doc in documents],
    documents=[doc["content"] for doc in documents],
    metadatas=[{
        "source": doc["source"],
        "page": doc["page"]
    } for doc in documents]
)

logger.info(f"Added {len(documents)} documents to ChromaDB")
print(f"Vector store ready with {collection.count()} documents")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

04:06:33 | INFO | Added 6 documents to ChromaDB
Vector store ready with 6 documents


LLM Setup

In [6]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    api_key=os.environ["GROQ_API_KEY"]
)

SYSTEM_PROMPT = """You are DocuMind, a precise document assistant.
Answer questions ONLY based on the provided context.
If the context does not contain enough information, say "I don't have enough information in my documents to answer this."
Always be specific and cite the key facts from the context.
Do not add information from outside the provided context."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", """Context from documents:
{context}

Question: {question}

Provide a precise answer based only on the context above.""")
])

chain = prompt | llm
logger.info("LLM initialized")

04:06:59 | INFO | LLM initialized


RAG Core Functions

In [8]:


def retrieve_documents(question: str, n_results: int = 3) -> dict:
    results = collection.query(
        query_texts=[question],
        n_results=n_results
    )

    chunks = []
    for i in range(len(results["documents"][0])):
        chunks.append({
            "content": results["documents"][0][i],
            "source": results["metadatas"][0][i]["source"],
            "page": results["metadatas"][0][i]["page"],
            "distance": results["distances"][0][i]
        })

    return chunks


def calculate_confidence(distances: list) -> float:
    if not distances:
        return 0.0
    avg_distance = sum(distances) / len(distances)
    confidence = max(0.0, min(1.0, 1 - avg_distance))
    return round(confidence, 2)


def format_context(chunks: list) -> str:
    context_parts = []
    for i, chunk in enumerate(chunks):
        context_parts.append(
            f"[Source {i+1}: {chunk['source']} Page {chunk['page']}]\n{chunk['content']}"
        )
    return "\n\n".join(context_parts)


def ask_documind(question: str) -> dict:
    logger.info(f"Question: {question}")

    # Step 1 — Retrieve relevant chunks
    chunks = retrieve_documents(question, n_results=3)
    logger.info(f"Retrieved {len(chunks)} chunks")

    # Step 2 — Format context
    context = format_context(chunks)

    # Step 3 — Generate answer
    response = chain.invoke({
        "context": context,
        "question": question
    })

    answer = response.content

    # Step 4 — Build citations
    citations = []
    for chunk in chunks:
        citations.append({
            "source": chunk["source"],
            "page": chunk["page"],
            "relevance_distance": round(chunk["distance"], 4)
        })

    # Step 5 — Calculate confidence
    distances = [chunk["distance"] for chunk in chunks]
    confidence = calculate_confidence(distances)

    logger.info(f"Answer generated | Confidence: {confidence}")

    return {
        "question": question,
        "answer": answer,
        "citations": citations,
        "confidence": confidence,
        "chunks_retrieved": len(chunks)
    }

logger.info("RAG functions defined")

04:07:07 | INFO | RAG functions defined


Test Single Question

In [9]:
result = ask_documind("What is RAG and how does it reduce hallucinations?")

print("\n========== DOCUMIND RESULT ==========")
print(f"Question   : {result['question']}")
print(f"Confidence : {result['confidence']}")
print(f"Chunks     : {result['chunks_retrieved']}")
print(f"\nAnswer:\n{result['answer']}")
print(f"\nCitations:")
for i, cite in enumerate(result['citations']):
    print(f"  [{i+1}] {cite['source']} — Page {cite['page']} (distance: {cite['relevance_distance']})")

04:07:18 | INFO | Question: What is RAG and how does it reduce hallucinations?
04:07:18 | INFO | Retrieved 3 chunks
04:07:18 | INFO | Answer generated | Confidence: 0.2

========== DOCUMIND RESULT ==========
Question   : What is RAG and how does it reduce hallucinations?
Confidence : 0.2
Chunks     : 3

Answer:
According to Source 1: RAG_Techniques.pdf Page 7, Retrieval Augmented Generation (RAG) combines information retrieval with text generation. It works by first searching a knowledge base for relevant documents and then using those documents as context for the Large Language Model (LLM) to generate accurate answers. RAG reduces hallucinations and improves factual accuracy significantly. 

In other words, RAG reduces hallucinations by using relevant documents from a knowledge base as context, allowing the LLM to generate more accurate and factual answers.

Citations:
  [1] RAG_Techniques.pdf — Page 7 (distance: 0.5512)
  [2] Vector_DB_Guide.pdf — Page 2 (distance: 0.8791)
  [3] LLM_

Test Multiple Questions

In [10]:
questions = [
    "What is the difference between AI and machine learning?",
    "How do vector databases work?",
    "What frameworks are used for agentic AI?",
    "What is prompt engineering?"
]

print("========== BATCH Q&A ==========\n")

for i, question in enumerate(questions):
    result = ask_documind(question)
    print(f"Q{i+1}: {question}")
    print(f"Answer: {result['answer'][:200]}...")
    print(f"Sources: {[c['source'] for c in result['citations']]}")
    print(f"Confidence: {result['confidence']}")
    print("---")

========== BATCH Q&A ==========

04:07:34 | INFO | Question: What is the difference between AI and machine learning?
04:07:34 | INFO | Retrieved 3 chunks
04:07:34 | INFO | Answer generated | Confidence: 0.52
Q1: What is the difference between AI and machine learning?
Answer: According to [Source 1: AI_Fundamentals.pdf Page 1], the difference between AI and machine learning is that "Artificial Intelligence (AI) is the simulation of human intelligence in machines programmed...
Sources: ['AI_Fundamentals.pdf', 'Agentic_AI.pdf', 'LLM_Guide.pdf']
Confidence: 0.52
---
04:07:34 | INFO | Question: How do vector databases work?
04:07:34 | INFO | Retrieved 3 chunks
04:07:35 | INFO | Answer generated | Confidence: 0.4
Q2: How do vector databases work?
Answer: According to Source 1: Vector_DB_Guide.pdf Page 2, vector databases "store data as high-dimensional vectors called embeddings" and "enable semantic search — finding similar content based on meaning ra...
Sources: ['Vector_DB_Guide.pdf', 'LLM

Test Unanswerable Question

In [11]:
result = ask_documind("What is the recipe for chocolate cake?")

print("========== UNANSWERABLE QUESTION TEST ==========")
print(f"Question   : {result['question']}")
print(f"Answer     : {result['answer']}")
print(f"Confidence : {result['confidence']}")
print(f"Citations  : {result['citations']}")

04:07:54 | INFO | Question: What is the recipe for chocolate cake?
04:07:54 | INFO | Retrieved 3 chunks
04:07:54 | INFO | Answer generated | Confidence: 0.08
========== UNANSWERABLE QUESTION TEST ==========
Question   : What is the recipe for chocolate cake?
Answer     : I don't have enough information in my documents to answer this.
Confidence : 0.08
Citations  : [{'source': 'AI_Fundamentals.pdf', 'page': 1, 'relevance_distance': 0.865}, {'source': 'Agentic_AI.pdf', 'page': 5, 'relevance_distance': 0.9342}, {'source': 'Prompt_Engineering.pdf', 'page': 4, 'relevance_distance': 0.9464}]


Project Summary

In [12]:
print("========== DOCUMIND SUMMARY ==========\n")
print("Project     : DocuMind — RAG Agent with Citation Grounding")
print("Author      : K Murali Krishna")
print("Model       : Groq LLaMA-3.3-70b-versatile")
print("Vector DB   : ChromaDB")
print("Embeddings  : sentence-transformers all-MiniLM-L6-v2")
print("\nKey Capabilities:")
print("  ✓ Semantic document search using vector embeddings")
print("  ✓ Citation grounding — every answer has a source")
print("  ✓ Confidence scoring based on retrieval distance")
print("  ✓ Handles unanswerable questions gracefully")
print("  ✓ Batch question processing")
print("\nProduction Concepts Demonstrated:")
print("  ✓ RAG pipeline — retrieve then generate")
print("  ✓ Vector similarity search")
print("  ✓ Hallucination reduction through context grounding")
print("  ✓ Citation tracking for enterprise trust")

========== DOCUMIND SUMMARY ==========

Project     : DocuMind — RAG Agent with Citation Grounding
Author      : K Murali Krishna
Model       : Groq LLaMA-3.3-70b-versatile
Vector DB   : ChromaDB
Embeddings  : sentence-transformers all-MiniLM-L6-v2

Key Capabilities:
  ✓ Semantic document search using vector embeddings
  ✓ Citation grounding — every answer has a source
  ✓ Confidence scoring based on retrieval distance
  ✓ Handles unanswerable questions gracefully
  ✓ Batch question processing

Production Concepts Demonstrated:
  ✓ RAG pipeline — retrieve then generate
  ✓ Vector similarity search
  ✓ Hallucination reduction through context grounding
  ✓ Citation tracking for enterprise trust
